# 贪心解码（Greedy Decoding）

> 每步选概率最大的 token，简单高效但容易陷入局部最优。

## 背景
自回归生成最简单的策略：每步取 argmax(logits)。优点是确定性、快速，
缺点是容易生成重复、缺乏多样性，且可能错过全局更优的序列。

## 公式
$$y_t = \arg\max_{y} P(y | y_{<t}, x)$$

## 复杂度
- 时间：O(T × V)，T=生成长度，V=词表大小
- 空间：O(1)，每步只保留 1 个 token
- vs Beam Search：贪心是 beam_size=1 的特例

## 考察点
- 重复问题：贪心容易生成重复序列
- 与采样对比：贪心确定性但缺乏多样性
- 适用场景：翻译/摘要等确定性任务


In [ ]:
import torch
import torch.nn.functional as F

def greedy_decode(model, input_ids, max_new_tokens=20):
    """贪心解码：每步选概率最高的 token。"""
    with torch.no_grad():
        logits = model(input_ids)
        next_token = torch.argmax(logits[0, -1], dim=-1, keepdim=True)
        generated = [next_token.item()]
        for _ in range(max_new_tokens - 1):
            logits = model(torch.tensor([generated[-1:]], dtype=torch.long))
            next_token = torch.argmax(logits[0, -1], dim=-1, keepdim=True)
            generated.append(next_token.item())
    return generated

# 简单验证: argmax(logits) 返回最大索引
logits = torch.tensor([[2.0, 5.0, 1.0, 3.0]])
result = torch.argmax(logits, dim=-1)
assert result.item() == 1, f"argmax should be 1, got {result.item()}"
print(f"✅ GreedyDecoding: argmax={result.item()}, 贪心解码函数已实现")


## 小结
- 贪心 = 每步 argmax，是 beam=1 的特例。
- 实际生成几乎不单独用贪心（除非评测/确定性输出），多配 TopK/TopP/温度。
- 与 KV cache 配合时 prefill 一次、之后每步只输入 1 token。

## ✅ 测试验证

In [ ]:
# 验证贪心解码
import torch
import torch.nn.functional as F

# 贪心解码: 每步选概率最大的 token
# 性质: 确定性（相同输入永远输出相同结果）
logits = torch.randn(5, 100)  # 5 步, vocab=100

# 贪心 = argmax
greedy_tokens = logits.argmax(dim=-1)
assert greedy_tokens.shape == (5,), f"shape wrong: {greedy_tokens.shape}"

# 确定性
greedy_tokens2 = logits.argmax(dim=-1)
assert torch.equal(greedy_tokens, greedy_tokens2), "greedy should be deterministic"

# 对比: 贪心选出的 token 概率应最大
probs = F.softmax(logits, dim=-1)
for t, step in enumerate(greedy_tokens):
    assert probs[t, step] == probs[t].max(), "greedy token should have max prob"

print("✅ GreedyDecoding 测试通过: 确定性、选最大概率 token")
